<a href="https://colab.research.google.com/github/samueltuiransierra-cmyk/PROG_CIVIL_2026_1/blob/main/An%C3%A1lisis_Hidr%C3%A1ulico_de_Canal_de_Riego.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CANAL DE RIEGO — Análisis Hidráulico con Python**.

Análisis Hidráulico de Canal de Riego

Curso: Programación de Computadores con Python

Ingeniería Civil

ESTUDIANTES:

SAMUEL TUIRAN SIERRA   1104261565

CARLOS JOSÉ MONTERROZA BUELVAS

LUIS MARIO MENCO GAVIRIA

## **1. Funciones Geométricas**
Se crearon las funciones `calcular_area`, `calcular_perimetro` y `calcular_ancho_superficial` para un canal trapezoidal.

In [1]:
import math

def calcular_area(b, z, y):
    """
    Calcula el área hidráulica de una sección transversal de canal trapezoidal.

    Args:
        b (float): Ancho de plantilla del canal (m).
        z (float): Talud de las paredes del canal (relación H:V).
        y (float): Tirante de agua en el canal (m).

    Returns:
        float: Área hidráulica (m²).
    """
    # A = (b + z*y) * y
    return (b + z * y) * y

def calcular_perimetro(b, z, y):
    """
    Calcula el perímetro mojado de una sección transversal de canal trapezoidal.

    Args:
        b (float): Ancho de plantilla del canal (m).
        z (float): Talud de las paredes del canal (relación H:V).
        y (float): Tirante de agua en el canal (m).

    Returns:
        float: Perímetro mojado (m).
    """
    # P = b + 2 * y * sqrt(1 + z^2)
    return b + 2 * y * math.sqrt(1 + z**2)

def calcular_ancho_superficial(b, z, y):
    """
    Calcula el ancho superficial de una sección transversal de canal trapezoidal.

    Args:
        b (float): Ancho de plantilla del canal (m).
        z (float): Talud de las paredes del canal (relación H:V).
        y (float): Tirante de agua en el canal (m).

    Returns:
        float: Ancho superficial (m).
    """
    # T = b + 2 * z * y
    return b + 2 * z * y

## **2. Función de Caudal y Velocidad**
Se implementó `calcular_caudal` utilizando la fórmula de Manning y las funciones geométricas, incluyendo validación de parámetros positivos.

In [11]:
def calcular_caudal(b, z, y, n, S):
    """
    Calcula el caudal (Q) y la velocidad (V) en un canal trapezoidal utilizando la fórmula de Manning.

    Args:
        b (float): Ancho de plantilla del canal (m).
        z (float): Talud de las paredes del canal (relación H:V).
        y (float): Tirante de agua en el canal (m).
        n (float): Coeficiente de rugosidad de Manning.
        S (float): Pendiente del fondo del canal (m/m).

    Returns:
        tuple: Una tupla que contiene (Q, V) donde Q es el caudal (m³/s) y V es la velocidad (m/s).
               Retorna (None, None) si algún parámetro es no positivo.
    """
    # Verificación de parámetros positivos
    if not all(p > 0 for p in [b, z, y, n, S]):
        print("Advertencia: Todos los parámetros de entrada deben ser positivos.")
        return None, None

    # Calcular área y perímetro mojado
    A = calcular_area(b, z, y)
    P = calcular_perimetro(b, z, y)

    # Calcular radio hidráulico
    if P == 0:
        print("Advertencia: El perímetro mojado es cero, no se puede calcular el radio hidráulico.")
        return None, None
    R = A / P

    # Fórmula de Manning para el caudal (Q = (1/n) * A * R^(2/3) * S^(1/2))
    Q = (1 / n) * A * (R**(2/3)) * (S**(1/2))

    # Caudal Q = A * V
    V = Q / A

    return Q, V

## **3. Clasificación del Régimen**
Se definió `clasificar_regimen` para categorizar el flujo como subcrítico, crítico o supercrítico basado en el número de Froude.

In [3]:
def clasificar_regimen(Fr):
    """
    Clasifica el régimen de flujo basado en el número de Froude (Fr).

    Args:
        Fr (float): Número de Froude.

    Returns:
        str: Un string describiendo el régimen de flujo y una advertencia si es supercrítico.
    """
    if Fr < 1:
        return "Subcrítico"
    elif Fr == 1:
        return "Crítico"
    else: # Fr > 1
        return "Supercrítico (Advertencia: Flujo inestable)"

## **4. Análisis por Rango de Tirantes**
Se creó una función auxiliar `calcular_numero_froude` y la función `analizar_canal` para iterar sobre un rango de tirantes, calcular todos los parámetros hidráulicos y almacenarlos.

In [4]:
def calcular_numero_froude(V, A, T):
    """
    Calcula el número de Froude para un canal.

    Args:
        V (float): Velocidad media del flujo (m/s).
        A (float): Área hidráulica (m²).
        T (float): Ancho superficial (m).

    Returns:
        float: Número de Froude.
    """
    g = 9.81 # Aceleración de la gravedad (m/s²)
    if T == 0:
        return float('inf') # Evitar división por cero, o indicar flujo crítico/supercrítico
    D = A / T # Profundidad hidráulica
    if D <= 0:
        return float('inf')
    Fr = V / math.sqrt(g * D)
    return Fr

def analizar_canal(b, z, n, S, y_min, y_max, paso):
    """
    Analiza las características hidráulicas de un canal trapezoidal para un rango de tirantes.

    Args:
        b (float): Ancho de plantilla del canal (m).
        z (float): Talud de las paredes del canal (relación H:V).
        n (float): Coeficiente de rugosidad de Manning.
        S (float): Pendiente del fondo del canal (m/m).
        y_min (float): Tirante mínimo a analizar (m).
        y_max (float): Tirante máximo a analizar (m).
        paso (float): Incremento del tirante en cada iteración (m).

    Returns:
        list: Una lista de diccionarios, donde cada diccionario contiene los resultados
              (y, A, P, R, Q, V, Fr, Régimen) para un tirante dado.
    """
    resultados = []
    y_actual = y_min

    while y_actual <= y_max:
        if y_actual <= 0: # Evitar tirantes no físicos
            y_actual += paso
            continue

        A = calcular_area(b, z, y_actual)
        P = calcular_perimetro(b, z, y_actual)
        T = calcular_ancho_superficial(b, z, y_actual)

        if P > 0:
            R = A / P
        else:
            R = 0

        Q, V = calcular_caudal(b, z, y_actual, n, S)

        Fr = None
        regimen = ""
        if V is not None and A > 0 and T > 0:
            Fr = calcular_numero_froude(V, A, T)
            regimen = clasificar_regimen(Fr)

        resultados.append({
            "y": round(y_actual, 3),
            "A": round(A, 3),
            "P": round(P, 3),
            "R": round(R, 3),
            "Q": round(Q, 4) if Q is not None else None,
            "V": round(V, 3) if V is not None else None,
            "Fr": round(Fr, 3) if Fr is not None else None,
            "Régimen": regimen
        })
        y_actual += paso

    return resultados

## **5. Demanda Agroindustrial**
Se añadieron `calcular_demanda_cultivo` para convertir la demanda a m³/s y `verificar_abastecimiento` para comparar el caudal del canal con las demandas de los cultivos.

In [5]:
def calcular_demanda_cultivo(nombre, ETo, Kc, area_ha):
    """
    Calcula la demanda de agua de un cultivo en m³/s.

    Args:
        nombre (str): Nombre del cultivo.
        ETo (float): Evapotranspiración de referencia (mm/día).
        Kc (float): Coeficiente de cultivo.
        area_ha (float): Área del cultivo en hectáreas.

    Returns:
        dict: Un diccionario con el nombre del cultivo y su demanda en m³/s.
    """
    # 1. Calcular la evapotranspiración del cultivo (ETc) en mm/día
    ETc_mm_dia = ETo * Kc

    # 2. Convertir ETc a metros por día
    ETc_m_dia = ETc_mm_dia / 1000 # mm a m

    # 3. Convertir área de hectáreas a metros cuadrados
    area_m2 = area_ha * 10000 # 1 hectárea = 10,000 m²

    # 4. Calcular el volumen de agua requerido por día (m³/día)
    volumen_m3_dia = ETc_m_dia * area_m2

    # 5. Convertir el volumen diario a caudal en m³/s
    # 1 día = 24 horas * 60 minutos * 60 segundos = 86400 segundos
    demanda_m3_s = volumen_m3_dia / 86400

    return {"nombre": nombre, "demanda_m3_s": demanda_m3_s}

def verificar_abastecimiento(Q_canal, cultivos):
    """
    Verifica si el caudal del canal es suficiente para abastecer cada cultivo.

    Args:
        Q_canal (float): Caudal disponible en el canal (m³/s).
        cultivos (list): Una lista de diccionarios, donde cada diccionario representa un cultivo
                         y contiene la clave 'demanda_m3_s'.

    Returns:
        list: Una lista de diccionarios con el nombre del cultivo y un mensaje de abastecimiento.
    """
    resultados_abastecimiento = []
    for cultivo in cultivos:
        nombre = cultivo["nombre"]
        demanda = cultivo["demanda_m3_s"]
        if Q_canal >= demanda:
            resultados_abastecimiento.append({"cultivo": nombre, "estado": "Abastecido"})
        else:
            resultados_abastecimiento.append({"cultivo": nombre, "estado": f"No abastecido (Faltan {demanda - Q_canal:.4f} m³/s)"})
    return resultados_abastecimiento

## **6. Programa Principal**
Se configuró el programa principal para ejecutar todas las funciones, imprimir una tabla de resultados y mostrar el estado de abastecimiento de los cultivos.

In [20]:
import math

# --- Paso 6: Programa principal ---

# 3. Valores de entrada y resultados de validación
# Parámetros del canal
b = 0.80        # ancho de plantilla (m)
z = 1.5           # talud, relación H:V
n = 0.014         # coeficiente de rugosidad de Manning (concreto liso)
S = 0.0008        # pendiente m/m
y_min = 0.20      # tirante mínimo (m)
y_max = 1.20      # tirante máximo (m)
paso = 0.20       # incremento del tirante (m)

print("\n--- Análisis Hidráulico del Canal ---")
print(f"Parámetros del canal: b={b}m, z={z}, n={n}, S={S}")
print(f"Rango de tirantes: y_min={y_min}m, y_max={y_max}m, paso={paso}m\n")

# Llamar a la función analizar_canal
resultados_analisis = analizar_canal(b, z, n, S, y_min, y_max, paso)

# Imprimir tabla de resultados
print("| {y:<6} | {A:<8} | {P:<8} | {R:<8} | {Q:<10} | {V:<8} | {Fr:<8} | {Regimen:<15} |".format(
    y='y (m)', A='A (m²)', P='P (m)', R='R (m)', Q='Q (m³/s)', V='V (m/s)', Fr='Fr', Regimen='Régimen'
))
print("|:-------|:---------|:---------|:---------|:-----------|:---------|:---------|:----------------|")
for res in resultados_analisis:
    print("| {y:<6.2f} | {A:<8.3f} | {P:<8.3f} | {R:<8.3f} | {Q:<10.4f} | {V:<8.3f} | {Fr:<8.3f} | {Regimen:<15} |".format(
        y=res['y'], A=res['A'], P=res['P'], R=res['R'], Q=res['Q'], V=res['V'], Fr=res['Fr'], Regimen=res['Régimen']
    ))

# --- Demanda Agroindustrial ---
print("\n--- Análisis de Demanda Agroindustrial ---")

# Datos de cultivos de ejemplo (actualizados según tu última instrucción)
cultivos_ejemplo_params = [
    {"nombre": "Maíz", "ETo": 5.2, "Kc": 1.05, "area_ha": 40},
    {"nombre": "Arroz", "ETo": 6.0, "Kc": 1.20, "area_ha": 350},
    {"nombre": "Palma", "ETo": 5.5, "Kc": 1.10, "area_ha": 600},
    {"nombre": "Caña", "ETo": 5.8, "Kc": 1.25, "area_ha": 800}
]

# Calcular demanda para cada cultivo
demandas_cultivos = []
for params in cultivos_ejemplo_params:
    demanda = calcular_demanda_cultivo(params["nombre"], params["ETo"], params["Kc"], params["area_ha"])
    demandas_cultivos.append(demanda)
    print(f"Demanda para {demanda['nombre']}: {demanda['demanda_m3_s']:.4f} m³/s")

# El usuario solicitó evaluar el abastecimiento con un Q fijo de 0.663 m³/s
Q_canal_para_verificacion = 0.663

print(f"\nCaudal del canal usado para verificación (valor fijo): {Q_canal_para_verificacion:.4f} m³/s")

# Verificar abastecimiento
print("\n--- Verificación de Abastecimiento ---")
resultados_abastecimiento = verificar_abastecimiento(Q_canal_para_verificacion, demandas_cultivos)
for res in resultados_abastecimiento:
    print(f"Cultivo: {res['cultivo']} - Estado: {res['estado']}")



--- Análisis Hidráulico del Canal ---
Parámetros del canal: b=0.8m, z=1.5, n=0.014, S=0.0008
Rango de tirantes: y_min=0.2m, y_max=1.2m, paso=0.2m

| y (m)  | A (m²)   | P (m)    | R (m)    | Q (m³/s)   | V (m/s)  | Fr       | Régimen         |
|:-------|:---------|:---------|:---------|:-----------|:---------|:---------|:----------------|
| 0.20   | 0.220    | 1.521    | 0.145    | 0.1225     | 0.557    | 0.448    | Subcrítico      |
| 0.40   | 0.560    | 2.242    | 0.250    | 0.4487     | 0.801    | 0.483    | Subcrítico      |
| 0.60   | 1.020    | 2.963    | 0.344    | 1.0121     | 0.992    | 0.506    | Subcrítico      |
| 0.80   | 1.600    | 3.684    | 0.434    | 1.8537     | 1.159    | 0.523    | Subcrítico      |
| 1.00   | 2.300    | 4.406    | 0.522    | 3.0128     | 1.310    | 0.538    | Subcrítico      |
| 1.20   | 3.120    | 5.127    | 0.609    | 4.5267     | 1.451    | 0.550    | Subcrítico      |

--- Análisis de Demanda Agroindustrial ---
Demanda para Maíz: 0.0253 m³/s
D